In [ ]:
# ============================================================
# Cell 0: Notebook Overview
# ============================================================
#
# Notebook:
# 08_Validate_and_Tune_Model.ipynb
#
# Purpose:
# Evaluate, compare, and tune Multi-Layer Perceptron (MLP)
# classifiers using the validation dataset in order to select
# the best-performing model configuration.
#
# This notebook focuses on model comparison, validation-based
# selection, and hyperparameter tuning.
#
# ------------------------------------------------------------
# Inputs:
# ------------------------------------------------------------
# Inputs from earlier pipeline stages:
#
# From Notebook 06:
# - Normalized training feature matrix
# - Normalized validation feature matrix
# - Encoded training labels
# - Encoded validation labels
#
# From Notebook 07:
# - Baseline model results summary
# - Previously trained baseline model files (.pkl), available
#   for reference and comparison
#
# ------------------------------------------------------------
# Assumptions:
# ------------------------------------------------------------
# - Baseline models have already been trained and saved
# - Training and validation data have been normalized using the
#   training-set scaler
# - The validation set is used only for model selection and
#   hyperparameter tuning
# - The test set remains completely untouched at this stage
#
# ------------------------------------------------------------
# Main Steps:
# ------------------------------------------------------------
# 1. Load normalized training and validation datasets
# 2. Load baseline model results for reference
# 3. Define tuned candidate MLP configurations by varying
#    selected hyperparameters (for example: alpha,
#    early_stopping, and/or hidden-layer structure)
# 4. Train each tuned candidate model on the training dataset
# 5. Run inference for each tuned model on the validation set
# 6. Compute validation metrics for each model:
#     - accuracy
#     - precision
#     - recall
#     - F1-score
#     - ROC AUC
# 7. Compare baseline and tuned model configurations in
#    tabular form
# 8. Identify the best-performing model configuration based on
#    validation results
# 9. Save the selected final model configuration and/or tuned
#    model artifact for use in Notebook 09
#
# ------------------------------------------------------------
# Outputs:
# ------------------------------------------------------------
# - Validation metrics for each tuned candidate model
# - Combined model comparison table
# - Selected best-performing model configuration
# - Saved tuned model file and/or selected configuration record
#   for Notebook 09
#
# ------------------------------------------------------------
# Notes:
# ------------------------------------------------------------
# - This notebook is the model selection stage of the pipeline
# - Hyperparameter tuning is performed using validation results
#   only
# - Baseline models from Notebook 07 are used as reference
#   points, not modified in place
# - The test dataset is NOT used at any point in this notebook
# - After model selection is complete, Notebook 09 will retrain
#   the final chosen configuration using the combined
#   train+validation dataset and a newly fitted scaler before
#   final evaluation on the test set
#
# ============================================================



In [1]:
# ============================================================
# Cell 1: Environment Setup and Imports
# ============================================================

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

# ------------------------------------------------------------
# Standard library imports
# ------------------------------------------------------------
import os
import json
import joblib
import warnings

# ------------------------------------------------------------
# Third-party imports
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.preprocessing import LabelEncoder

# ------------------------------------------------------------
# Optional display settings
# ------------------------------------------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------
BASE_DRIVE_DIR = "/content/drive/MyDrive/DIP_Project"

# Normalized feature-vector inputs from Notebook 06
TRAIN_DATA_PATH = os.path.join(BASE_DRIVE_DIR, "train_feature_vectors_normalized.csv")
VALIDATION_DATA_PATH = os.path.join(BASE_DRIVE_DIR, "validation_feature_vectors_normalized.csv")

# Baseline results from Notebook 07
BASELINE_RESULTS_PATH = os.path.join(BASE_DRIVE_DIR, "baseline_model_results.csv")

# Model directories
MODEL_INPUT_DIR = os.path.join(BASE_DRIVE_DIR, "models")
MODEL_OUTPUT_DIR = os.path.join(BASE_DRIVE_DIR, "models")
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

# Output files for Notebook 08
TUNED_RESULTS_CSV_PATH = os.path.join(BASE_DRIVE_DIR, "tuned_model_results.csv")
BEST_MODEL_CONFIG_PATH = os.path.join(BASE_DRIVE_DIR, "best_model_config.json")

print("Environment setup complete.")
print(f"Base project directory    : {BASE_DRIVE_DIR}")
print(f"Training data path        : {TRAIN_DATA_PATH}")
print(f"Validation data path      : {VALIDATION_DATA_PATH}")
print(f"Baseline results path     : {BASELINE_RESULTS_PATH}")
print(f"Model directory           : {MODEL_INPUT_DIR}")
print(f"Tuned results output path : {TUNED_RESULTS_CSV_PATH}")
print(f"Best config output path   : {BEST_MODEL_CONFIG_PATH}")


Mounted at /content/drive
Environment setup complete.
Base project directory    : /content/drive/MyDrive/DIP_Project
Training data path        : /content/drive/MyDrive/DIP_Project/train_feature_vectors_normalized.csv
Validation data path      : /content/drive/MyDrive/DIP_Project/validation_feature_vectors_normalized.csv
Baseline results path     : /content/drive/MyDrive/DIP_Project/baseline_model_results.csv
Model directory           : /content/drive/MyDrive/DIP_Project/models
Tuned results output path : /content/drive/MyDrive/DIP_Project/tuned_model_results.csv
Best config output path   : /content/drive/MyDrive/DIP_Project/best_model_config.json


In [2]:
# ============================================================
# Cell 2: Load Normalized Data and Baseline Results
# ============================================================

print("Loading normalized feature-vector datasets and baseline results...\n")

# ------------------------------------------------------------
# Load normalized training and validation datasets
# ------------------------------------------------------------
train_df = pd.read_csv(TRAIN_DATA_PATH)
validation_df = pd.read_csv(VALIDATION_DATA_PATH)

print("Loaded normalized datasets:")
print(f"Train      : {TRAIN_DATA_PATH}")
print(f"Validation : {VALIDATION_DATA_PATH}")

print("\nDataset shapes:")
print(f"train_df      : {train_df.shape}")
print(f"validation_df : {validation_df.shape}")

# ------------------------------------------------------------
# Define non-feature columns
# ------------------------------------------------------------
NON_FEATURE_COLUMNS = [
    "filename",
    "class_label",
    "source_dataset",
    "subset"
]

non_feature_cols_present = [col for col in NON_FEATURE_COLUMNS if col in train_df.columns]

# ------------------------------------------------------------
# Separate features and labels
# ------------------------------------------------------------
X_train_df = train_df.drop(columns=non_feature_cols_present)
X_validation_df = validation_df.drop(columns=non_feature_cols_present)

y_train_raw = train_df["class_label"].copy()
y_validation_raw = validation_df["class_label"].copy()

# ------------------------------------------------------------
# Convert to numpy arrays
# ------------------------------------------------------------
X_train = X_train_df.values
X_validation = X_validation_df.values

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_validation = label_encoder.transform(y_validation_raw)

# ------------------------------------------------------------
# Load baseline results summary from Notebook 07
# ------------------------------------------------------------
baseline_results_df = pd.read_csv(BASELINE_RESULTS_PATH)

print("\nLoaded baseline results:")
print(f"Baseline results : {BASELINE_RESULTS_PATH}")
print(f"baseline_results_df shape : {baseline_results_df.shape}")

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------
print("\nFeature columns used:")
print(list(X_train_df.columns))

print("\nFinal shapes:")
print(f"X_train            : {X_train.shape}")
print(f"X_validation       : {X_validation.shape}")
print(f"y_train            : {y_train.shape}")
print(f"y_validation       : {y_validation.shape}")

print("\nLabel mapping:")
for idx, label in enumerate(label_encoder.classes_):
    print(f"  {label} -> {idx}")

print("\nClass distribution (train):")
print(pd.Series(y_train_raw).value_counts())

print("\nClass distribution (validation):")
print(pd.Series(y_validation_raw).value_counts())

print("\nBaseline model summary:")
display(baseline_results_df)# ============================================================
# Cell 2: Load Normalized Data and Baseline Results
# ============================================================

print("Loading normalized feature-vector datasets and baseline results...\n")

# ------------------------------------------------------------
# Load normalized training and validation datasets
# ------------------------------------------------------------
train_df = pd.read_csv(TRAIN_DATA_PATH)
validation_df = pd.read_csv(VALIDATION_DATA_PATH)

print("Loaded normalized datasets:")
print(f"Train      : {TRAIN_DATA_PATH}")
print(f"Validation : {VALIDATION_DATA_PATH}")

print("\nDataset shapes:")
print(f"train_df      : {train_df.shape}")
print(f"validation_df : {validation_df.shape}")

# ------------------------------------------------------------
# Define non-feature columns
# ------------------------------------------------------------
NON_FEATURE_COLUMNS = [
    "filename",
    "class_label",
    "source_dataset",
    "subset"
]

non_feature_cols_present = [col for col in NON_FEATURE_COLUMNS if col in train_df.columns]

# ------------------------------------------------------------
# Separate features and labels
# ------------------------------------------------------------
X_train_df = train_df.drop(columns=non_feature_cols_present)
X_validation_df = validation_df.drop(columns=non_feature_cols_present)

y_train_raw = train_df["class_label"].copy()
y_validation_raw = validation_df["class_label"].copy()

# ------------------------------------------------------------
# Convert to numpy arrays
# ------------------------------------------------------------
X_train = X_train_df.values
X_validation = X_validation_df.values

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_validation = label_encoder.transform(y_validation_raw)

# ------------------------------------------------------------
# Load baseline results summary from Notebook 07
# ------------------------------------------------------------
baseline_results_df = pd.read_csv(BASELINE_RESULTS_PATH)

print("\nLoaded baseline results:")
print(f"Baseline results : {BASELINE_RESULTS_PATH}")
print(f"baseline_results_df shape : {baseline_results_df.shape}")

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------
print("\nFeature columns used:")
print(list(X_train_df.columns))

print("\nFinal shapes:")
print(f"X_train            : {X_train.shape}")
print(f"X_validation       : {X_validation.shape}")
print(f"y_train            : {y_train.shape}")
print(f"y_validation       : {y_validation.shape}")

print("\nLabel mapping:")
for idx, label in enumerate(label_encoder.classes_):
    print(f"  {label} -> {idx}")

print("\nClass distribution (train):")
print(pd.Series(y_train_raw).value_counts())

print("\nClass distribution (validation):")
print(pd.Series(y_validation_raw).value_counts())

print("\nBaseline model summary:")
display(baseline_results_df)


Loading normalized feature-vector datasets and baseline results...

Loaded normalized datasets:
Train      : /content/drive/MyDrive/DIP_Project/train_feature_vectors_normalized.csv
Validation : /content/drive/MyDrive/DIP_Project/validation_feature_vectors_normalized.csv

Dataset shapes:
train_df      : (8400, 30)
validation_df : (1800, 30)

Loaded baseline results:
Baseline results : /content/drive/MyDrive/DIP_Project/baseline_model_results.csv
baseline_results_df shape : (3, 7)

Feature columns used:
['Mean Gradient', 'Std Gradient', 'Max Gradient', 'Gradient Entropy', 'Edge Density', 'Orientation Mean', 'Orientation Std', 'Orientation Entropy', 'Global Entropy', 'Local Entropy Mean', 'Local Entropy Std', 'Intensity Mean', 'Intensity Std', 'Laplacian Variance', 'Patch Variance Mean', 'Patch Variance Std', 'Noise Residual Energy', 'Low Frequency Energy Ratio', 'Mid Frequency Energy Ratio', 'High Frequency Energy Ratio', 'Radial Mean', 'Radial Std', 'Radial Entropy', 'Spectral Centroid'

,model_name,hidden_layer_sizes,train_accuracy,validation_accuracy,final_training_loss,n_iterations,model_path
0,MLP_Large,"(128, 64)",0.996905,0.880556,0.022265,231,/content/drive/MyDrive/DIP_Project/models/MLP_...
1,MLP_Medium,"(64, 32)",0.980238,0.872222,0.072627,300,/content/drive/MyDrive/DIP_Project/models/MLP_...
2,MLP_Small,"(32,)",0.901667,0.870556,0.233440,300,/content/drive/MyDrive/DIP_Project/models/MLP_...


Loading normalized feature-vector datasets and baseline results...

Loaded normalized datasets:
Train      : /content/drive/MyDrive/DIP_Project/train_feature_vectors_normalized.csv
Validation : /content/drive/MyDrive/DIP_Project/validation_feature_vectors_normalized.csv

Dataset shapes:
train_df      : (8400, 30)
validation_df : (1800, 30)

Loaded baseline results:
Baseline results : /content/drive/MyDrive/DIP_Project/baseline_model_results.csv
baseline_results_df shape : (3, 7)

Feature columns used:
['Mean Gradient', 'Std Gradient', 'Max Gradient', 'Gradient Entropy', 'Edge Density', 'Orientation Mean', 'Orientation Std', 'Orientation Entropy', 'Global Entropy', 'Local Entropy Mean', 'Local Entropy Std', 'Intensity Mean', 'Intensity Std', 'Laplacian Variance', 'Patch Variance Mean', 'Patch Variance Std', 'Noise Residual Energy', 'Low Frequency Energy Ratio', 'Mid Frequency Energy Ratio', 'High Frequency Energy Ratio', 'Radial Mean', 'Radial Std', 'Radial Entropy', 'Spectral Centroid'

,model_name,hidden_layer_sizes,train_accuracy,validation_accuracy,final_training_loss,n_iterations,model_path
0,MLP_Large,"(128, 64)",0.996905,0.880556,0.022265,231,/content/drive/MyDrive/DIP_Project/models/MLP_...
1,MLP_Medium,"(64, 32)",0.980238,0.872222,0.072627,300,/content/drive/MyDrive/DIP_Project/models/MLP_...
2,MLP_Small,"(32,)",0.901667,0.870556,0.233440,300,/content/drive/MyDrive/DIP_Project/models/MLP_...


In [3]:
# ============================================================
# Cell 3: Sanity Checks and Baseline Reference Confirmation
# ============================================================

print("Running sanity checks...\n")

# ------------------------------------------------------------
# Shape checks
# ------------------------------------------------------------
assert X_train.ndim == 2, "X_train must be a 2D feature matrix"
assert X_validation.ndim == 2, "X_validation must be a 2D feature matrix"
assert y_train.ndim == 1, "y_train must be a 1D label array"
assert y_validation.ndim == 1, "y_validation must be a 1D label array"

assert X_train.shape[0] == y_train.shape[0], "Mismatch between X_train and y_train row counts"
assert X_validation.shape[0] == y_validation.shape[0], "Mismatch between X_validation and y_validation row counts"
assert X_train.shape[1] == X_validation.shape[1], "Train and validation feature counts do not match"

# ------------------------------------------------------------
# Missing and finite value checks
# ------------------------------------------------------------
assert not np.isnan(X_train).any(), "NaN values found in X_train"
assert not np.isnan(X_validation).any(), "NaN values found in X_validation"
assert np.isfinite(X_train).all(), "Non-finite values found in X_train"
assert np.isfinite(X_validation).all(), "Non-finite values found in X_validation"

# ------------------------------------------------------------
# Class checks
# ------------------------------------------------------------
train_classes = np.unique(y_train)
validation_classes = np.unique(y_validation)

print("Unique classes in y_train      :", train_classes)
print("Unique classes in y_validation :", validation_classes)

assert len(train_classes) == 2, "Expected exactly 2 classes in y_train"
assert len(validation_classes) == 2, "Expected exactly 2 classes in y_validation"

# ------------------------------------------------------------
# Normalization spot check
# ------------------------------------------------------------
train_means = np.mean(X_train, axis=0)
train_stds = np.std(X_train, axis=0)

print("\nNormalization spot check (training set):")
print("Mean (first 5 features):")
print(train_means[:5])

print("\nStd (first 5 features):")
print(train_stds[:5])

# ------------------------------------------------------------
# Baseline results checks
# ------------------------------------------------------------
required_baseline_columns = [
    "model_name",
    "hidden_layer_sizes",
    "train_accuracy",
    "validation_accuracy",
    "final_training_loss",
    "n_iterations",
    "model_path"
]

missing_cols = [col for col in required_baseline_columns if col not in baseline_results_df.columns]
assert len(missing_cols) == 0, f"Missing columns in baseline_results_df: {missing_cols}"

assert len(baseline_results_df) == 3, "Expected 3 baseline models in baseline_results_df"

print("\nBaseline models available:")
print(list(baseline_results_df["model_name"]))

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
print("\nSanity check summary:")
print(f"Training samples       : {X_train.shape[0]}")
print(f"Validation samples     : {X_validation.shape[0]}")
print(f"Number of features     : {X_train.shape[1]}")
print(f"Training label counts  : {pd.Series(y_train).value_counts().to_dict()}")
print(f"Validation label counts: {pd.Series(y_validation).value_counts().to_dict()}")

print("\nAll sanity checks passed.")


Running sanity checks...

Unique classes in y_train      : [0 1]
Unique classes in y_validation : [0 1]

Normalization spot check (training set):
Mean (first 5 features):
[-2.23947844e-16 -2.91618581e-16  3.04941257e-16  3.79379068e-16
 -2.45306421e-17]

Std (first 5 features):
[1. 1. 1. 1. 1.]

Baseline models available:
['MLP_Large', 'MLP_Medium', 'MLP_Small']

Sanity check summary:
Training samples       : 8400
Validation samples     : 1800
Number of features     : 26
Training label counts  : {0: 4200, 1: 4200}
Validation label counts: {0: 900, 1: 900}

All sanity checks passed.


In [4]:
# ============================================================
# Cell 4: Define Tuned MLP Model Configurations
# ============================================================

# ------------------------------------------------------------
# Rationale:
# ------------------------------------------------------------
# Based on baseline results from Notebook 07:
# - Larger models show strong training performance but exhibit
#   overfitting (large train/validation gap)
#
# Tuning strategy:
# - Apply stronger regularization (alpha)
# - Enable early stopping to prevent over-training
# - Keep architectures similar to strong baseline models
# ------------------------------------------------------------

# Base configuration (shared)
MLP_BASE_CONFIG = {
    "activation": "relu",
    "solver": "adam",
    "batch_size": "auto",
    "learning_rate": "constant",
    "learning_rate_init": 0.001,
    "max_iter": 300,
    "random_state": 42
}

# Tuned configurations
TUNED_MODEL_CONFIGS = {
    "MLP_Large_alpha_0.001": {
        "hidden_layer_sizes": (128, 64),
        "alpha": 0.001,
        "early_stopping": False
    },
    "MLP_Large_alpha_0.01": {
        "hidden_layer_sizes": (128, 64),
        "alpha": 0.01,
        "early_stopping": False
    },
    "MLP_Large_early_stop": {
        "hidden_layer_sizes": (128, 64),
        "alpha": 0.0001,
        "early_stopping": True
    },
    "MLP_Medium_alpha_0.001": {
        "hidden_layer_sizes": (64, 32),
        "alpha": 0.001,
        "early_stopping": False
    }
}

# Build models
TUNED_MODELS = {}

for model_name, overrides in TUNED_MODEL_CONFIGS.items():
    config = MLP_BASE_CONFIG.copy()
    config.update(overrides)
    TUNED_MODELS[model_name] = MLPClassifier(**config)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------
print("Tuned models defined:\n")

for model_name, model in TUNED_MODELS.items():
    print(f"{model_name}:")
    print(f"  hidden_layer_sizes = {model.hidden_layer_sizes}")
    print(f"  alpha              = {model.alpha}")
    print(f"  early_stopping     = {model.early_stopping}")
    print()


Tuned models defined:

MLP_Large_alpha_0.001:
  hidden_layer_sizes = (128, 64)
  alpha              = 0.001
  early_stopping     = False

MLP_Large_alpha_0.01:
  hidden_layer_sizes = (128, 64)
  alpha              = 0.01
  early_stopping     = False

MLP_Large_early_stop:
  hidden_layer_sizes = (128, 64)
  alpha              = 0.0001
  early_stopping     = True

MLP_Medium_alpha_0.001:
  hidden_layer_sizes = (64, 32)
  alpha              = 0.001
  early_stopping     = False



In [5]:
# ============================================================
# Cell 5: Train Tuned Models and Evaluate on Validation Set
# ============================================================

# ------------------------------------------------------------
# Purpose:
# ------------------------------------------------------------
# Train each tuned MLP model on the normalized training dataset,
# evaluate performance on the validation dataset, and compute
# the full set of validation metrics needed for model selection.
# ------------------------------------------------------------

tuned_results = []

print("Training tuned models...\n")

for model_name, model in TUNED_MODELS.items():
    print(f"Training {model_name} ...")

    # --------------------------------------------------------
    # Train model
    # --------------------------------------------------------
    model.fit(X_train, y_train)

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------
    y_train_pred = model.predict(X_train)
    y_validation_pred = model.predict(X_validation)

    # Probability scores for ROC AUC
    y_validation_prob = model.predict_proba(X_validation)[:, 1]

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------
    train_accuracy = accuracy_score(y_train, y_train_pred)
    validation_accuracy = accuracy_score(y_validation, y_validation_pred)
    validation_precision = precision_score(y_validation, y_validation_pred)
    validation_recall = recall_score(y_validation, y_validation_pred)
    validation_f1 = f1_score(y_validation, y_validation_pred)
    validation_roc_auc = roc_auc_score(y_validation, y_validation_prob)

    final_training_loss = model.loss_

    # --------------------------------------------------------
    # Save tuned model
    # --------------------------------------------------------
    model_path = os.path.join(MODEL_OUTPUT_DIR, f"{model_name}.pkl")
    joblib.dump(model, model_path)

    # --------------------------------------------------------
    # Store summary
    # --------------------------------------------------------
    tuned_results.append({
        "model_name": model_name,
        "hidden_layer_sizes": model.hidden_layer_sizes,
        "alpha": model.alpha,
        "early_stopping": model.early_stopping,
        "train_accuracy": train_accuracy,
        "validation_accuracy": validation_accuracy,
        "validation_precision": validation_precision,
        "validation_recall": validation_recall,
        "validation_f1": validation_f1,
        "validation_roc_auc": validation_roc_auc,
        "final_training_loss": final_training_loss,
        "n_iterations": model.n_iter_,
        "model_path": model_path
    })

    # --------------------------------------------------------
    # Status print
    # --------------------------------------------------------
    print(f"  hidden_layer_sizes   : {model.hidden_layer_sizes}")
    print(f"  alpha                : {model.alpha}")
    print(f"  early_stopping       : {model.early_stopping}")
    print(f"  train_accuracy       : {train_accuracy:.4f}")
    print(f"  validation_accuracy  : {validation_accuracy:.4f}")
    print(f"  validation_precision : {validation_precision:.4f}")
    print(f"  validation_recall    : {validation_recall:.4f}")
    print(f"  validation_f1        : {validation_f1:.4f}")
    print(f"  validation_roc_auc   : {validation_roc_auc:.4f}")
    print(f"  final_training_loss  : {final_training_loss:.6f}")
    print(f"  n_iterations         : {model.n_iter_}")
    print(f"  saved_to             : {model_path}\n")

print("Tuned model training complete.")


Training tuned models...

Training MLP_Large_alpha_0.001 ...
  hidden_layer_sizes   : (128, 64)
  alpha                : 0.001
  early_stopping       : False
  train_accuracy       : 0.9988
  validation_accuracy  : 0.8906
  validation_precision : 0.8759
  validation_recall    : 0.9100
  validation_f1        : 0.8926
  validation_roc_auc   : 0.9513
  final_training_loss  : 0.020711
  n_iterations         : 231
  saved_to             : /content/drive/MyDrive/DIP_Project/models/MLP_Large_alpha_0.001.pkl

Training MLP_Large_alpha_0.01 ...
  hidden_layer_sizes   : (128, 64)
  alpha                : 0.01
  early_stopping       : False
  train_accuracy       : 0.9985
  validation_accuracy  : 0.8767
  validation_precision : 0.8792
  validation_recall    : 0.8733
  validation_f1        : 0.8763
  validation_roc_auc   : 0.9485
  final_training_loss  : 0.037393
  n_iterations         : 232
  saved_to             : /content/drive/MyDrive/DIP_Project/models/MLP_Large_alpha_0.01.pkl

Training MLP_La

In [6]:
# ============================================================
# Cell 6: Summarize Tuned Model Results
# ============================================================

# Convert collected tuned results to a DataFrame
tuned_results_df = pd.DataFrame(tuned_results)

# Sort by validation accuracy, then F1-score, then ROC AUC
tuned_results_df = tuned_results_df.sort_values(
    by=["validation_accuracy", "validation_f1", "validation_roc_auc"],
    ascending=False
).reset_index(drop=True)

print("Tuned model summary:\n")
display(tuned_results_df)


Tuned model summary:



,model_name,hidden_layer_sizes,alpha,early_stopping,train_accuracy,validation_accuracy,validation_precision,validation_recall,validation_f1,validation_roc_auc,final_training_loss,n_iterations,model_path
0,MLP_Large_alpha_0.001,"(128, 64)",0.0010,False,0.998810,0.890556,0.875936,0.910000,0.892643,0.951283,0.020711,231,/content/drive/MyDrive/DIP_Project/models/MLP_...
1,MLP_Large_early_stop,"(128, 64)",0.0001,True,0.925595,0.880556,0.863203,0.904444,0.883342,0.952616,0.169963,50,/content/drive/MyDrive/DIP_Project/models/MLP_...
2,MLP_Large_alpha_0.01,"(128, 64)",0.0100,False,0.998452,0.876667,0.879195,0.873333,0.876254,0.948521,0.037393,232,/content/drive/MyDrive/DIP_Project/models/MLP_...
3,MLP_Medium_alpha_0.001,"(64, 32)",0.0010,False,0.980238,0.876111,0.878212,0.873333,0.875766,0.949695,0.074967,300,/content/drive/MyDrive/DIP_Project/models/MLP_...


In [7]:
# ============================================================
# Cell 7: Combine Baseline and Tuned Model Results
# ============================================================

# ------------------------------------------------------------
# Purpose:
# ------------------------------------------------------------
# Create a unified comparison table containing both baseline
# and tuned model results. Missing fields are added so the
# tables can be compared consistently.
# ------------------------------------------------------------

# Copy baseline results and add missing columns for alignment
baseline_compare_df = baseline_results_df.copy()
baseline_compare_df["alpha"] = np.nan
baseline_compare_df["early_stopping"] = np.nan
baseline_compare_df["validation_precision"] = np.nan
baseline_compare_df["validation_recall"] = np.nan
baseline_compare_df["validation_f1"] = np.nan
baseline_compare_df["validation_roc_auc"] = np.nan
baseline_compare_df["model_type"] = "baseline"

# Copy tuned results and add model type
tuned_compare_df = tuned_results_df.copy()
tuned_compare_df["model_type"] = "tuned"

# Align column order
comparison_columns = [
    "model_type",
    "model_name",
    "hidden_layer_sizes",
    "alpha",
    "early_stopping",
    "train_accuracy",
    "validation_accuracy",
    "validation_precision",
    "validation_recall",
    "validation_f1",
    "validation_roc_auc",
    "final_training_loss",
    "n_iterations",
    "model_path"
]

baseline_compare_df = baseline_compare_df[comparison_columns]
tuned_compare_df = tuned_compare_df[comparison_columns]

# Combine and sort
all_results_df = pd.concat([baseline_compare_df, tuned_compare_df], ignore_index=True)
all_results_df = all_results_df.sort_values(
    by=["validation_accuracy", "validation_f1", "validation_roc_auc"],
    ascending=False
).reset_index(drop=True)

print("Combined baseline + tuned model comparison:\n")
display(all_results_df)


Combined baseline + tuned model comparison:



,model_type,model_name,hidden_layer_sizes,alpha,early_stopping,train_accuracy,validation_accuracy,validation_precision,validation_recall,validation_f1,validation_roc_auc,final_training_loss,n_iterations,model_path
0,tuned,MLP_Large_alpha_0.001,"(128, 64)",0.0010,0.0,0.998810,0.890556,0.875936,0.910000,0.892643,0.951283,0.020711,231,/content/drive/MyDrive/DIP_Project/models/MLP_...
1,tuned,MLP_Large_early_stop,"(128, 64)",0.0001,1.0,0.925595,0.880556,0.863203,0.904444,0.883342,0.952616,0.169963,50,/content/drive/MyDrive/DIP_Project/models/MLP_...
2,baseline,MLP_Large,"(128, 64)",NaN,NaN,0.996905,0.880556,NaN,NaN,NaN,NaN,0.022265,231,/content/drive/MyDrive/DIP_Project/models/MLP_...
3,tuned,MLP_Large_alpha_0.01,"(128, 64)",0.0100,0.0,0.998452,0.876667,0.879195,0.873333,0.876254,0.948521,0.037393,232,/content/drive/MyDrive/DIP_Project/models/MLP_...
4,tuned,MLP_Medium_alpha_0.001,"(64, 32)",0.0010,0.0,0.980238,0.876111,0.878212,0.873333,0.875766,0.949695,0.074967,300,/content/drive/MyDrive/DIP_Project/models/MLP_...
5,baseline,MLP_Medium,"(64, 32)",NaN,NaN,0.980238,0.872222,NaN,NaN,NaN,NaN,0.072627,300,/content/drive/MyDrive/DIP_Project/models/MLP_...
6,baseline,MLP_Small,"(32,)",NaN,NaN,0.901667,0.870556,NaN,NaN,NaN,NaN,0.233440,300,/content/drive/MyDrive/DIP_Project/models/MLP_...


In [9]:
# ============================================================
# Cell 8: Select Best Model and Save Final Configuration
# ============================================================

import ast

# ------------------------------------------------------------
# Selection rule:
# ------------------------------------------------------------
# Choose the best tuned model based primarily on validation
# accuracy, with F1-score and ROC AUC available as additional
# supporting metrics.
# ------------------------------------------------------------

best_model_row = tuned_results_df.iloc[0]

best_model_name = best_model_row["model_name"]
best_model_path = best_model_row["model_path"]

print("Best tuned model selected:\n")
print(f"Model name           : {best_model_name}")
print(f"Validation accuracy  : {best_model_row['validation_accuracy']:.4f}")
print(f"Validation precision : {best_model_row['validation_precision']:.4f}")
print(f"Validation recall    : {best_model_row['validation_recall']:.4f}")
print(f"Validation F1-score  : {best_model_row['validation_f1']:.4f}")
print(f"Validation ROC AUC   : {best_model_row['validation_roc_auc']:.4f}")
print(f"Saved model path     : {best_model_path}")

# ------------------------------------------------------------
# Safely recover hidden-layer structure
# ------------------------------------------------------------
hidden_layers = best_model_row["hidden_layer_sizes"]

if isinstance(hidden_layers, str):
    hidden_layers = ast.literal_eval(hidden_layers)

hidden_layers = list(hidden_layers)

# ------------------------------------------------------------
# Build configuration record for Notebook 09
# ------------------------------------------------------------
best_model_config = {
    "model_name": best_model_name,
    "hidden_layer_sizes": hidden_layers,
    "activation": "relu",
    "solver": "adam",
    "alpha": float(best_model_row["alpha"]),
    "batch_size": "auto",
    "learning_rate": "constant",
    "learning_rate_init": 0.001,
    "max_iter": 300,
    "random_state": 42,
    "early_stopping": bool(best_model_row["early_stopping"])
}

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------
with open(BEST_MODEL_CONFIG_PATH, "w") as f:
    json.dump(best_model_config, f, indent=4)

tuned_results_df.to_csv(TUNED_RESULTS_CSV_PATH, index=False)

print("\nSaved outputs:")
print(f"Tuned results CSV    : {TUNED_RESULTS_CSV_PATH}")
print(f"Best model config    : {BEST_MODEL_CONFIG_PATH}")

print("\nBest model configuration:")
print(json.dumps(best_model_config, indent=4))


Best tuned model selected:

Model name           : MLP_Large_alpha_0.001
Validation accuracy  : 0.8906
Validation precision : 0.8759
Validation recall    : 0.9100
Validation F1-score  : 0.8926
Validation ROC AUC   : 0.9513
Saved model path     : /content/drive/MyDrive/DIP_Project/models/MLP_Large_alpha_0.001.pkl

Saved outputs:
Tuned results CSV    : /content/drive/MyDrive/DIP_Project/tuned_model_results.csv
Best model config    : /content/drive/MyDrive/DIP_Project/best_model_config.json

Best model configuration:
{
    "model_name": "MLP_Large_alpha_0.001",
    "hidden_layer_sizes": [
        128,
        64
    ],
    "activation": "relu",
    "solver": "adam",
    "alpha": 0.001,
    "batch_size": "auto",
    "learning_rate": "constant",
    "learning_rate_init": 0.001,
    "max_iter": 300,
    "random_state": 42,
    "early_stopping": false
}
